In [0]:
%pip install transformers accelerate sentence-transformers chromadb

dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import chromadb
from typing import List, Dict
import re

In [0]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# mistralai/Mistral-7B-Instruct

In [0]:
VECTOR_DB_PATH = "/local_disk0/tmp/chroma_db"

import chromadb

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection(
    name="legal_knowledge"
)

print("Collection ready")

In [0]:
import os
os.listdir("/local_disk0/tmp/chroma_db")

In [0]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_length=512,
    temperature=0.2
)

In [0]:
def embed_query(query: str):
    return embedding_model.encode([query]).tolist()[0]

In [0]:
def retrieve_chunks(query: str, k: int = 10):
    query_vector = embed_query(query)

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=k
    )

    documents = results["documents"][0]
    metadata = results["metadatas"][0]

    return documents, metadata

In [0]:
def rerank_and_filter(query: str, docs: List[str], metadata: List[Dict]):
    filtered = []

    query_terms = set(query.lower().split())

    for doc, meta in zip(docs, metadata):
        text = doc.lower()

        # keyword relevance scoring
        score = sum(1 for term in query_terms if term in text)

        # prioritize sections with penalties
        if "penalty" in text or "fine" in text or "punishable" in text:
            score += 2

        # prioritize exact legal references
        if meta.get("section"):
            score += 1

        filtered.append((score, doc, meta))

    filtered.sort(reverse=True, key=lambda x: x[0])

    # return top relevant
    return filtered[:5]

In [0]:
def build_context(filtered_results):
    unique_docs = []
    seen = set()

    for _, doc, _ in filtered_results:
        snippet = doc.strip()

        if snippet not in seen and len(snippet) > 120:
            unique_docs.append(snippet)
            seen.add(snippet)

    return unique_docs[:3]

In [0]:
def build_legal_prompt(query: str, context: List[str]):

    context_text = "\n\n".join(context)

    prompt = f"""
You are an AI legal assistant helping Indian citizens understand laws.

Answer clearly and in simple language.

Include:
• relevant law
• penalties or consequences
• citizen guidance
• when applicable, exceptions

Question:
{query}

Legal Context:
{context_text}

Provide a structured and helpful answer.
"""

    return prompt

In [0]:
def generate_legal_answer(query: str):
    docs, metadata = retrieve_chunks(query)

    ranked = rerank_and_filter(query, docs, metadata)

    context = build_context(ranked)

    prompt = build_legal_prompt(query, context)

    response = llm(prompt)[0]["generated_text"]

    # extract sections
    sections = {
        meta.get("section")
        for _, _, meta in ranked
        if meta.get("section")
    }

    return response, sections

In [0]:
def format_legal_output(answer: str, sections):

    section_text = ", ".join(sections) if sections else "Applicable provisions"

    return f"""
⚖️ LEGAL EXPLANATION

{answer}

📚 Relevant Legal Sections:
{section_text}

🧾 Practical Advice:
Follow the law to avoid penalties and ensure safety.

⚠️ DISCLAIMER:
This AI-generated response provides legal information for educational purposes and is not a substitute for professional legal advice.
"""

In [0]:
def ask_legal_assistant(query: str):
    answer, sections = generate_legal_answer(query)
    return format_legal_output(answer, sections)

In [0]:
print(ask_legal_assistant(
    "What is the penalty for not wearing a helmet in India?"
))